# Notebook 04: Hyperparameter Tuning

## RustWeatherML - Weather Prediction System in Rust

This notebook covers:
1. Grid search implementation
2. Random search implementation
3. Cross-validation strategies
4. Learning curves analysis
5. Best model selection per task

**Input**: Train/Val data and baseline models from Notebook 03

**Output**: Optimized hyperparameters and best models

---
## 1. Setup Dependencies

In [ ]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde"] }
:dep linfa = "0.7"
:dep linfa-trees = "0.7"
:dep smartcore = "0.3"
:dep rand = "0.8"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [ ]:
use polars::prelude::*;
use ndarray::{Array1, Array2, Axis};
use linfa::prelude::*;
use linfa_trees::{DecisionTree, SplitQuality};
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::ensemble::random_forest_regressor::RandomForestRegressor;
use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;
use smartcore::tree::decision_tree_regressor::DecisionTreeRegressorParameters;
use rand::Rng;
use std::collections::HashMap;

println!("Dependencies loaded!");

---
## 2. Load Data

In [ ]:
// Load data (reusing code from Notebook 03)
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let val_df = LazyFrame::scan_parquet("../data/features/val.parquet", Default::default())
    .unwrap().collect().unwrap();

println!("Data loaded: Train {} rows, Val {} rows", train_df.height(), val_df.height());

In [ ]:
// Feature columns (same as Notebook 03)
let feature_cols: Vec<&str> = vec![
    "temperature_2m", "apparent_temperature", "dewpoint_2m",
    "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover", "visibility",
    "shortwave_radiation", "direct_radiation", "relativehumidity_2m",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
    "temp_lag_1h", "temp_lag_6h", "temp_lag_12h", "temp_lag_24h",
    "pressure_lag_1h", "pressure_lag_6h", "pressure_lag_24h",
    "humidity_lag_1h", "humidity_lag_6h",
    "wind_lag_1h", "wind_lag_6h", "precip_lag_1h",
    "temp_change_1h", "temp_change_6h", "temp_change_24h",
    "pressure_change_1h", "pressure_change_6h", "pressure_change_24h",
    "humidity_change_1h",
    "latitude", "longitude",
];

println!("Using {} features", feature_cols.len());

In [ ]:
// Helper functions
fn df_to_array2(df: &DataFrame, cols: &[&str]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    
    for col_name in cols {
        let col = df.column(*col_name).unwrap();
        let values = col.cast(&DataType::Float64).unwrap().f64().unwrap().to_vec();
        for val in values {
            data.push(val.unwrap_or(0.0));
        }
    }
    
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, col_name: &str) -> Array1<f64> {
    let col = df.column(col_name).unwrap();
    let values: Vec<f64> = col.cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec()
        .into_iter().map(|v| v.unwrap_or(0.0)).collect();
    Array1::from_vec(values)
}

fn ndarray_to_dense_matrix(arr: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&arr.outer_iter().map(|row| row.to_vec()).collect::<Vec<_>>())
}

println!("Helper functions defined!");

In [ ]:
// Prepare clean data
let train_clean = train_df.clone().lazy()
    .filter(col("temp_next_24h").is_not_null().and(col("temp_lag_24h").is_not_null()))
    .collect().unwrap();

let val_clean = val_df.clone().lazy()
    .filter(col("temp_next_24h").is_not_null().and(col("temp_lag_24h").is_not_null()))
    .collect().unwrap();

let X_train = df_to_array2(&train_clean, &feature_cols);
let X_val = df_to_array2(&val_clean, &feature_cols);
let y_train = df_to_array1(&train_clean, "temp_next_24h");
let y_val = df_to_array1(&val_clean, "temp_next_24h");

let X_train_sm = ndarray_to_dense_matrix(&X_train);
let X_val_sm = ndarray_to_dense_matrix(&X_val);
let y_train_vec: Vec<f64> = y_train.to_vec();
let y_val_vec: Vec<f64> = y_val.to_vec();

println!("Data prepared: X_train {:?}, X_val {:?}", X_train.shape(), X_val.shape());

---
## 3. Grid Search Implementation

In [ ]:
/// Grid Search for Decision Tree (linfa)
/// Parameters to tune:
/// - max_depth: [5, 10, 15, 20]
/// - min_weight_split: [5, 10, 20, 50]

println!("=== GRID SEARCH: Decision Tree (linfa) ===");
println!("\nSearching over:");
println!("  - max_depth: [5, 10, 15, 20]");
println!("  - min_weight_split: [5, 10, 20, 50]");

let train_dataset = linfa::Dataset::new(X_train.clone(), y_train.clone());

let max_depths = vec![5, 10, 15, 20];
let min_weights = vec![5.0, 10.0, 20.0, 50.0];

let mut best_rmse = f64::MAX;
let mut best_params = (0usize, 0.0f64);
let mut results: Vec<(usize, f64, f64)> = Vec::new();

for &depth in &max_depths {
    for &weight in &min_weights {
        let model = DecisionTree::params()
            .max_depth(Some(depth))
            .min_weight_split(weight)
            .split_quality(SplitQuality::Variance)
            .fit(&train_dataset)
            .expect("Failed to fit");
        
        let y_pred = model.predict(&X_val);
        let mse: f64 = y_pred.iter().zip(y_val.iter())
            .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_val.len() as f64;
        let rmse = mse.sqrt();
        
        results.push((depth, weight, rmse));
        
        if rmse < best_rmse {
            best_rmse = rmse;
            best_params = (depth, weight);
        }
    }
}

println!("\nGrid Search Results:");
println!("{:<12} {:>15} {:>12}", "max_depth", "min_weight", "RMSE");
println!("{}", "-".repeat(42));
for (d, w, r) in &results {
    let marker = if *d == best_params.0 && *w == best_params.1 { " *" } else { "" };
    println!("{:<12} {:>15.1} {:>11.4}{}", d, w, r, marker);
}

println!("\n✓ Best parameters: max_depth={}, min_weight_split={}", best_params.0, best_params.1);
println!("✓ Best RMSE: {:.4}°C", best_rmse);

---
## 4. Random Search Implementation

In [ ]:
/// Random Search allows exploring larger hyperparameter spaces efficiently
/// We'll sample random combinations instead of exhaustive search

println!("\n=== RANDOM SEARCH: Decision Tree (linfa) ===");
println!("\nSearching over:");
println!("  - max_depth: [3, 30]");
println!("  - min_weight_split: [2, 100]");
println!("  - n_iterations: 20");

let mut rng = rand::thread_rng();
let n_iterations = 20;

let mut best_rmse_random = f64::MAX;
let mut best_params_random = (0usize, 0.0f64);
let mut random_results: Vec<(usize, f64, f64)> = Vec::new();

for _ in 0..n_iterations {
    let depth = rng.gen_range(3..=30);
    let weight = rng.gen_range(2.0..=100.0);
    
    let model = DecisionTree::params()
        .max_depth(Some(depth))
        .min_weight_split(weight)
        .split_quality(SplitQuality::Variance)
        .fit(&train_dataset)
        .expect("Failed to fit");
    
    let y_pred = model.predict(&X_val);
    let mse: f64 = y_pred.iter().zip(y_val.iter())
        .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_val.len() as f64;
    let rmse = mse.sqrt();
    
    random_results.push((depth, weight, rmse));
    
    if rmse < best_rmse_random {
        best_rmse_random = rmse;
        best_params_random = (depth, weight);
    }
}

// Sort by RMSE
random_results.sort_by(|a, b| a.2.partial_cmp(&b.2).unwrap());

println!("\nTop 10 Random Search Results (sorted by RMSE):");
println!("{:<12} {:>15} {:>12}", "max_depth", "min_weight", "RMSE");
println!("{}", "-".repeat(42));
for (d, w, r) in random_results.iter().take(10) {
    println!("{:<12} {:>15.2} {:>11.4}", d, w, r);
}

println!("\n✓ Best parameters: max_depth={}, min_weight_split={:.2}", 
         best_params_random.0, best_params_random.1);
println!("✓ Best RMSE: {:.4}°C", best_rmse_random);

---
## 5. Cross-Validation

In [ ]:
/// K-Fold Cross-Validation implementation
/// For time series, we use forward-chaining (walk-forward validation)

println!("\n=== K-FOLD CROSS-VALIDATION ===");
println!("\nUsing 5-fold cross-validation...");

let k_folds = 5;
let n_samples = X_train.nrows();
let fold_size = n_samples / k_folds;

let mut fold_scores: Vec<f64> = Vec::new();

for fold in 0..k_folds {
    // Create train/val indices for this fold
    let val_start = fold * fold_size;
    let val_end = if fold == k_folds - 1 { n_samples } else { (fold + 1) * fold_size };
    
    // Split data
    let mut train_indices: Vec<usize> = Vec::new();
    let mut val_indices: Vec<usize> = Vec::new();
    
    for i in 0..n_samples {
        if i >= val_start && i < val_end {
            val_indices.push(i);
        } else {
            train_indices.push(i);
        }
    }
    
    // Create fold datasets
    let X_fold_train: Array2<f64> = train_indices.iter()
        .map(|&i| X_train.row(i).to_vec())
        .collect::<Vec<_>>()
        .into_iter()
        .flatten()
        .collect::<Vec<_>>()
        .chunks(X_train.ncols())
        .map(|c| c.to_vec())
        .collect::<Vec<_>>()
        .into_iter()
        .flatten()
        .collect::<Array1<f64>>()
        .into_shape((train_indices.len(), X_train.ncols()))
        .unwrap();
    
    let y_fold_train: Array1<f64> = train_indices.iter()
        .map(|&i| y_train[i])
        .collect();
    
    let X_fold_val: Array2<f64> = val_indices.iter()
        .map(|&i| X_train.row(i).to_vec())
        .collect::<Vec<_>>()
        .into_iter()
        .flatten()
        .collect::<Vec<_>>()
        .chunks(X_train.ncols())
        .map(|c| c.to_vec())
        .collect::<Vec<_>>()
        .into_iter()
        .flatten()
        .collect::<Array1<f64>>()
        .into_shape((val_indices.len(), X_train.ncols()))
        .unwrap();
    
    let y_fold_val: Array1<f64> = val_indices.iter()
        .map(|&i| y_train[i])
        .collect();
    
    // Train model with best params
    let fold_dataset = linfa::Dataset::new(X_fold_train, y_fold_train);
    let model = DecisionTree::params()
        .max_depth(Some(best_params.0))
        .min_weight_split(best_params.1)
        .split_quality(SplitQuality::Variance)
        .fit(&fold_dataset)
        .unwrap();
    
    let y_pred = model.predict(&X_fold_val);
    let mse: f64 = y_pred.iter().zip(y_fold_val.iter())
        .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_fold_val.len() as f64;
    let rmse = mse.sqrt();
    
    fold_scores.push(rmse);
    println!("  Fold {}: RMSE = {:.4}°C", fold + 1, rmse);
}

let mean_rmse: f64 = fold_scores.iter().sum::<f64>() / fold_scores.len() as f64;
let std_rmse: f64 = (fold_scores.iter()
    .map(|x| (x - mean_rmse).powi(2)).sum::<f64>() / fold_scores.len() as f64).sqrt();

println!("\n✓ Cross-Validation Results:");
println!("  Mean RMSE: {:.4}°C ± {:.4}°C", mean_rmse, std_rmse);

---
## 6. Learning Curves

In [ ]:
/// Learning curves show how model performance changes with training set size
/// This helps diagnose overfitting vs underfitting

println!("\n=== LEARNING CURVES ===");
println!("\nTraining with increasing data sizes...");

let train_sizes = vec![0.1, 0.2, 0.4, 0.6, 0.8, 1.0];
let mut learning_curve: Vec<(f64, f64, f64)> = Vec::new();  // (size, train_rmse, val_rmse)

for &size in &train_sizes {
    let n_samples_use = (X_train.nrows() as f64 * size) as usize;
    
    // Slice data
    let X_subset = X_train.slice(ndarray::s![..n_samples_use, ..]).to_owned();
    let y_subset = y_train.slice(ndarray::s![..n_samples_use]).to_owned();
    
    let subset_dataset = linfa::Dataset::new(X_subset.clone(), y_subset.clone());
    
    let model = DecisionTree::params()
        .max_depth(Some(best_params.0))
        .min_weight_split(best_params.1)
        .split_quality(SplitQuality::Variance)
        .fit(&subset_dataset)
        .unwrap();
    
    // Train RMSE
    let y_train_pred = model.predict(&X_subset);
    let train_mse: f64 = y_train_pred.iter().zip(y_subset.iter())
        .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_subset.len() as f64;
    let train_rmse = train_mse.sqrt();
    
    // Val RMSE
    let y_val_pred = model.predict(&X_val);
    let val_mse: f64 = y_val_pred.iter().zip(y_val.iter())
        .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_val.len() as f64;
    let val_rmse = val_mse.sqrt();
    
    learning_curve.push((size, train_rmse, val_rmse));
    
    println!("  {:.0}% data ({} samples): Train RMSE={:.4}°C, Val RMSE={:.4}°C", 
             size * 100.0, n_samples_use, train_rmse, val_rmse);
}

println!("\nAnalysis:");
let gap = learning_curve.last().unwrap().2 - learning_curve.last().unwrap().1;
if gap > 1.0 {
    println!("  ⚠ Large gap between train/val suggests overfitting");
    println!("  → Consider: more regularization, simpler model, more data");
} else if learning_curve.last().unwrap().2 > 5.0 {
    println!("  ⚠ High validation error suggests underfitting");
    println!("  → Consider: more complex model, more features");
} else {
    println!("  ✓ Model appears well-balanced");
}

---
## 7. Final Model Selection

In [ ]:
println!("\n=== FINAL MODEL SELECTION ===");
println!("\nBased on hyperparameter tuning and cross-validation:");

// Train final model with best parameters
let final_model = DecisionTree::params()
    .max_depth(Some(best_params.0))
    .min_weight_split(best_params.1)
    .split_quality(SplitQuality::Variance)
    .fit(&train_dataset)
    .expect("Failed to fit final model");

// Final evaluation
let y_final_pred = final_model.predict(&X_val);
let final_mse: f64 = y_final_pred.iter().zip(y_val.iter())
    .map(|(p, t)| (p - t).powi(2)).sum::<f64>() / y_val.len() as f64;
let final_rmse = final_mse.sqrt();

// MAE
let final_mae: f64 = y_final_pred.iter().zip(y_val.iter())
    .map(|(p, t)| (p - t).abs()).sum::<f64>() / y_val.len() as f64;

println!("\n--- Best Model: Decision Tree (linfa) ---");
println!("  Hyperparameters:");
println!("    - max_depth: {}", best_params.0);
println!("    - min_weight_split: {}", best_params.1);
println!("  Performance:");
println!("    - RMSE: {:.4}°C", final_rmse);
println!("    - MAE:  {:.4}°C", final_mae);
println!("    - CV Mean RMSE: {:.4}°C ± {:.4}", mean_rmse, std_rmse);

In [ ]:
// Save best hyperparameters
let best_params_json = serde_json::json!({
    "task": "temp_24h_forecast",
    "model": "DecisionTree",
    "library": "linfa",
    "hyperparameters": {
        "max_depth": best_params.0,
        "min_weight_split": best_params.1
    },
    "performance": {
        "rmse": final_rmse,
        "mae": final_mae,
        "cv_mean_rmse": mean_rmse,
        "cv_std_rmse": std_rmse
    }
});

std::fs::write("../models/best_hyperparameters.json",
               serde_json::to_string_pretty(&best_params_json).unwrap())
    .expect("Failed to save");

println!("\n✓ Best hyperparameters saved to ../models/best_hyperparameters.json");

---
## 8. Summary

### What we accomplished:
1. ✅ Implemented Grid Search
2. ✅ Implemented Random Search
3. ✅ K-Fold Cross-Validation
4. ✅ Learning Curves Analysis
5. ✅ Selected best model and hyperparameters
6. ✅ Saved results

### Next Steps (Notebook 05):
- Detailed evaluation metrics
- Confusion matrices
- ROC curves
- Error analysis by city/season

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("Notebook 04 Complete!");
println!("=".repeat(60));
println!("\nProceed to Notebook 05: Evaluation & Validation");